# Inspecao Visual — Fase 6

Blink visual de cada candidato final.
Carregar da tabela `candidatos_fase5` no SQLite.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB = Path.cwd().parent / "data" / "rastro.db"
conn = sqlite3.connect(str(DB))

df = pd.read_sql("SELECT * FROM candidatos_fase5 ORDER BY llm_triage_score", conn)
print(f"Candidatos carregados: {len(df)}")
df[['source_id', 'ra', 'dec', 'pm_total', 'cor_w1w2', 'llm_triage_score']].head(20)

## Abrir no IRSA Finder Chart

Para cada candidato, abrir o IRSA Finder Chart para inspecao visual:
https://irsa.ipac.caltech.edu/applications/finderchart/

In [ ]:
from urllib.parse import quote

prioritarios = df[df['llm_triage_score'].isin(['alta', 'media'])]

for _, row in prioritarios.head(10).iterrows():
    url = f"https://irsa.ipac.caltech.edu/applications/finderchart/?ra={row['ra']}&dec={row['dec']}&size=1&band=1"
    print(f"source_id={row['source_id']}: {url}")

## Checklist de inspecao visual

Para cada candidato, verificar:
- [ ] Objeto visivel em multiplas epocas?
- [ ] Movimento proprio consistente com o PM medido?
- [ ] Proximo de estrela brilhante? (risco de fantasma de difracao)
- [ ] Forma pontual (nao extenso/artefato)?

Anotar em `human_review_notes`.

In [ ]:
# Adicionar coluna de revisao humana
if 'human_review_notes' not in df.columns:
    df['human_review_notes'] = None

# Exemplo de anotacao:
# df.at[0, 'human_review_notes'] = 'OK - pontual, PM consistente, sem estrela brilhante perto'
# df.at[0, 'status'] = 'candidato'  # ou 'descartado'

df.to_sql('candidatos_fase6', conn, if_exists='replace', index=False)
conn.close()